# DriveGuard — Step 2C: Fine-Tuning `MobileNetV3-Small` on the MRL Eye Dataset

This notebook can be run **locally in VS Code / Jupyter** OR uploaded to **Google Colab (with a free T4 GPU)** to train the 2-class eye-state classifier (`OPEN=0`, `CLOSED=1`) and export `mobilenetv3_eye.onnx` for DriveGuard's **Confidence-Gated Eye Cascade**.

### Two-Phase Transfer Learning Strategy:
1. **Phase 1 (Classifier Head Warmup):** Freeze `MobileNetV3-Small` feature extractor layers and train only the new 2-class linear head (`OPEN`, `CLOSED`).
2. **Phase 2 (Backbone Fine-Tuning):** Unfreeze the final 4 inverted-residual blocks of `MobileNetV3-Small` and fine-tune with a smaller learning rate (`1.5e-4`) and Cosine Annealing.
3. **Subject-Disjoint Evaluation:** Splits by MRL Subject ID (`s0001..s0037`) so no participant is shared between training and testing.
4. **ONNX Export:** Exports `mobilenetv3_eye.onnx` ready to drop into `backend/weights/mobilenetv3_eye.onnx`.

In [ ]:
# 1. Install PyTorch, TorchVision, and ONNX Runtime if not already installed
!pip install -q torch torchvision onnx onnxruntime opencv-python matplotlib

In [ ]:
import os
from pathlib import Path
import torch
import torchvision

print(f"PyTorch: {torch.__version__} | TorchVision: {torchvision.__version__} | CUDA GPU Available: {torch.cuda.is_available()}")

# Set this to the folder where you extracted the MRL Eye Dataset
MRL_DATA_DIR = Path(r"C:/Users/Michael Umoize/Desktop/DriveGuard/backend/datasets/data")

In [ ]:
# 2. Run the DriveGuard Transfer-Learning & ONNX Export Pipeline
import sys
repo_root = Path.cwd().parents[1] if Path.cwd().name == "research" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from backend.research.train_mobilenetv3_mrl import run_training_pipeline, create_synthetic_mrl_smoke_dataset

if MRL_DATA_DIR.is_dir():
    metrics = run_training_pipeline(
        data_dir=MRL_DATA_DIR,
        epochs_warmup=3,
        epochs_finetune=6,
        batch_size=64,
    )
else:
    print(f"MRL folder {MRL_DATA_DIR} not found yet — running quick smoke-test to verify pipeline & export ONNX...")
    smoke_dir = create_synthetic_mrl_smoke_dataset(repo_root / "backend" / "research" / "outputs", num_samples=200)
    metrics = run_training_pipeline(
        data_dir=smoke_dir,
        epochs_warmup=1,
        epochs_finetune=1,
        batch_size=32,
    )

print(metrics)

In [ ]:
# 3. Verify that DriveGuard's ConfidenceGatedEyeCascade automatically loads the exported ONNX model
from backend.app.ai.cascade_classifier import MobileNetV3EyeVerifier

onnx_path = repo_root / "backend" / "weights" / "mobilenetv3_eye.onnx"
verifier = MobileNetV3EyeVerifier(onnx_model_path=onnx_path)
print("ONNX Model Loaded into DriveGuard Cascade Verifier:", verifier.is_pretrained_onnx)